# 02 — LLaMEA Synthesis

**GPT-6 Luna token pilot · f10 · 10D · Explicit Baseline · heteroscedastic noise σ = 0.2**

Saved outputs may reflect earlier runs. Use separate study storage for an independent experiment.


In [5]:
import os
import sys
from pathlib import Path

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(root_dir)
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Keep module identities stable for multiprocessing/pickling.
# After updating project code or configuration, restart the kernel instead of evicting modules.

from evolution.application.model_selection import ModelSelection
from evolution.infra.di import EvolutionProvider
from evolution.application.campaign.audit import CampaignAudit
from evolution.application.campaign.service import SynthesisCampaign
from shared.infra.lifecycle import WorkflowScope

print("[OK] LLaMEA Evolution Pipeline initialized.")

# Select the provider and exact model for this campaign.
model = ModelSelection(provider="upb", model_id="gwdg.qwen3-coder-next")


[OK] LLaMEA Evolution Pipeline initialized.


## 1. Connect the model


In [6]:
# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(EvolutionProvider(), context={}, request_context={})
campaign = services.get(SynthesisCampaign)
audit = services.get(CampaignAudit)
print("=" * 65)
print(f"Provider: {model.provider}")
print(f"Model ID: {model.model_id}")
print("=" * 65)

Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/evolution/2026-11-23_14-42-28_UTC-0072ea76/diagnostics.log
Provider: upb
Model ID: gwdg.qwen3-coder-next


## 2. Audit synthesis coverage

Check the selected matrix against the current database before running.


In [7]:
matrix_df_result = audit.inspect(
    model=model,
    problem_targets=[{"id": 15, "dimensions": [50]}],
    noise_conditions=[{"std": 0.1, "noise_model": "heteroscedastic"}],
    synthesis_modes=[{"mode": "implicit", "strategies": ["guided"]}],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
)
if matrix_df_result.is_success:
    matrix_df, summary = matrix_df_result.value
    display(matrix_df) if "display" in globals() else print(matrix_df.to_string())
else:
    matrix_df = None
    summary = None
    print(f"Error: {matrix_df_result.error.message}")



🎯 Synthesis Matrix Audit for 'gwdg.qwen3-coder-next':
                                                   Target Runs  Completed  Running Records  Failed Records      Status
Problem         Dimension Environment    Strategy                                                                     
f15 (Rastrigin) 50D       Implicit (0.1) Guided              1          1                0               0  ✅ Complete
   • Configured Matrix Size:  1 conditions
   • Completed (Champions):   1/1 (100.0%)
   • Without Champion:       0


## 3. Run synthesis

Completed conditions are skipped; interrupted runs require their checkpoints to resume.


In [ ]:
# Rerun Qwen3 experiment #7154 to find a valid champion.
results_result = campaign.run(
    model=model,
    problem_targets=[],
    noise_conditions=[],
    synthesis_modes=[],
    iterations=10,
    budget=100_000,
    timeout_seconds=30.0,
    runs_per_config=1,
    auto_resume=True,
    skip_completed=True,
    retry_failed_synthesis=True,
    rerun_experiment_ids=[],
)
results = results_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)



🧬 LLAMEA EVOLUTIONARY ALGORITHM SYNTHESIS
   Model: gwdg.qwen3-coder-next | Pending Tasks: 1 | Concurrency: 8 workers
⚠️  Replacing Exp ID: #7154: old iterations, champion, code and checkpoint will be removed.

[1/1] 🧬 Task: gwdg.qwen3-coder-next | Dim: 50D | Noise: noisy (σ=0.1) | Problem: f15 (RastriginRotated) | Strategy: Guided | Exp ID: #7154
Gateway token usage for gwdg.qwen3-coder-next: input=1507 cached_input=None output=2342 reasoning=None total=3849 finish=stop
  • ⚡ Gen  1/10 | Algo: NoiseRobustES      | Error:   6.9853e+02 | Fitness:  -6.9853e+02 | Evals:  99,975 | Time:  1.22s
Gateway token usage for gwdg.qwen3-coder-next: input=3088 cached_input=None output=2575 reasoning=None total=5663 finish=stop
  • ⚡ Gen  2/10 | Algo: NoiseTunedES       | Error:   9.4226e+02 | Fitness:  -9.4226e+02 | Evals:  79,590 | Time:  0.96s
Gateway token usage for gwdg.qwen3-coder-next: input=3088 cached_input=None output=2575 reasoning=None total=5663 finish=stop
  • ⚡ Gen  3/10 | Algo: Noise

   Failed / Incomplete       : 0

